In [43]:
import torch
from torch.utils.data import Dataset

class SimpleDataset(Dataset):
    def __init__(self):
        self.x = torch.randn(100,10)
        self.y = torch.randint(0,2,(100,))
    def __len__(self):
        return len(self.x)
    def __getitem__(self,idx):
        return self.x[idx],self.y[idx]

ds = SimpleDataset()
print(f"样本数: {len(ds)}")
print(f"第0个样本: {ds[0][0].shape}, label={ds[0][1]}")




样本数: 100
第0个样本: torch.Size([10]), label=0


In [ ]:
from torch.utils.data import DataLoader

loader = DataLoer(
    dataset=ds,
    batch_size=32,#每批样本数
    shuffle=True,#每个epoch打乱顺序
    num_workers=4,#子进程数
    collate_fn=None,#自定义批合并函数
    drop_last=False,#丢弃末尾不足batch——size的批
    pin_memory=True,#GPU训练时开启
    persistent_worker=False,
)

In [45]:
import os 
import pandas as pd
import numpy as np
import torch
from torch.utils.data import Dataset,DataLoader
#生成测试csv
csv_path = 'test_data.csv'
if not os.path.exists(csv_path):
    np.random.seed(42)
    n_samples = 100
    df = pd.DataFrame({
        'age':np.random.randint(18,60,n_samples),
        'income':np.random.randn(n_samples)*10000 + 50000,
        'city':np.random.choice('beijing','shanghai','guangzhou',n_samples),
        'label':np.random.randint(0,2,n_samples)
    })
df.to_csv(csv_path,index=False)
print(f"✅ 已生成测试数据: {csv_path}")

#定义Dataset
class CSVDataset(Dataset):
    def __init__(self,csv_path,num_cols,cat_col,label_col):
        self.df = pd.read_csv(csv_path)
    #数值特征预提取为numpy
        self.num_features = self.df[num_cols].values.astype(np.float32)
        self.labels = self.df[label_col].values.astype(np.int64)
    #类别特征映射
        self.categories = sorted(self.df[cat_col].unique())
        self.cat_to_idx = {c:i for i,c in enumerate(self.categories)}
        self.cat_indices = self.df[cat_col].map(self.cat_to_idx).values.astype(np.int64)

        self.num_classes = len(self.categories)

    def __len__(self):
       return len(self.labels)

    def __getitem__(self,idx):
       x_num = torch.from_numpy(self.num_features[idx])
       x_cat = torch.tensor(self.cat_indices[idx],dtype=torch.long)
       y = torch.tensor(self.labels[idx],dtype=torch.long)
       return x_num,x_cat,y

#自定义collate_fn
def my_collate_fn(batch):
    x_nums,x_cats,ys=zip(*batch)#解包

    x_nums = torch.stack(x_nums)

    x_cats = torch.stack(x_cats)
    x_cat_onehot = torch.nn.functional.one_hot(x_cats,num_classes=3).float()
    
    x_final = torch.cat([x_nums,x_cat_onehot],dim=1)
    ys = torch.stack(ys)

    return x_final,ys

dataset = CSVDataset(
    csv_path = csv_path,
    num_cols=['age','income'],
    cat_col='city',
    label_col='label'
)

loader = DataLoader(
    dataset,
    batch_size=16,#每批16个样本
    shuffle=True,#训练集打乱
    num_workers=0,#
    collate_fn=my_collate_fn,#使用自定义的批处理函数
    drop_last=False,#丢弃末尾不足batch_size的批次
    pin_memory=True#GPU训练时开启，加速CPU->GPU传输
)

print(f"\n📊 数据集信息: {len(dataset)} 个样本, 类别映射: {dataset.cat_to_idx}")
print("-" * 50)

for batch_idx,(X_batch,y_batch) in enumerate(loader):
    print(f"Batch {batch_idx + 1}:")
    print(f"  特征 X shape: {X_batch.shape}, dtype: {X_batch.dtype}")
    print(f"  标签 y shape: {y_batch.shape}, dtype: {y_batch.dtype}")
    print(f"  样本0 特征: {X_batch[0].tolist()}")
    print(f"  样本0 标签: {y_batch[0].item()}")
    break

✅ 已生成测试数据: test_data.csv

📊 数据集信息: 100 个样本, 类别映射: {'Beijing': 0, 'Guangzhou': 1, 'Shanghai': 2}
--------------------------------------------------
Batch 1:
  特征 X shape: torch.Size([16, 5]), dtype: torch.float32
  标签 y shape: torch.Size([16]), dtype: torch.int64
  样本0 特征: [45.0, 48466.63671875, 0.0, 1.0, 0.0]
  样本0 标签: 1


In [ ]:
import torch
import torch.nn as nn
class MyFirstNet(nn.Module):
    def __init__(self):
        super().__init__()#等价于super(MyFirstNet, self).__init__()
        # ① 在 __init__ 中「声明」层（创建可学习参数）
        self.fc1 = nn.Linear(784,256)
        self.fc2 = nn.Linear(256,10)
        self.relu = nn.ReLU()

    def forward(self,x):
       # ② 在 forward 中「定义」数据如何流过这些层
        x = self.relu(self.fc1(x))
        y = self.fc2(x)
        return x

model = MyFirstNet()
dummy_input = torch.randn(32,784)#batch_size=32
output = model(dummy_input)#自动调用forward()
print(output.shape)#torch.Size([32,10])

torch.Size([32, 256])


In [26]:
class BadNet(nn.Module):
    def __init__(self):
        super().__init__()
        fc = nn.Linear(10,5)
        self.fc = nn.Linear(10,5)

    def forward(self,x):
        return self.fc(x)


In [42]:
import torch
import torch.nn as nn

class LeNet5(nn.Module):
    def __init__(self,num_classes=10):
        super(LeNet5,self).__init__()
    
        self.conv1 = nn.Conv2d(in_channels=1,out_channels=6,kernel_size=5)
        self.pool1 = nn.AvgPool2d(kernel_size=2,stride=2)
        self.conv2 = nn.Conv2d(in_channels=6,out_channels=16,kernel_size=5)
        self.pool2 = nn.AvgPool2d(kernel_size=2,stride=2)
        self.conv3 = nn.Conv2d(in_channels=16,out_channels=120,kernel_size=5)

        self.fc1 = nn.Linear(120,84)
        self.fc2 = nn.Linear(84,num_classes)
    
    def forward(self,x):

        x = torch.relu(self.conv1(x))
        print(f"  C1 Conv + ReLU  → {list(x.shape)}")

        x = self.pool1(x)
        print(f"  S2 AvgPool      → {list(x.shape)}")

        x = torch.relu(self.conv2(x))
        print(f"  C3 Conv + ReLU  → {list(x.shape)}")

        x = self.pool2(x)
        print(f"  S4 AvgPool      → {list(x.shape)}")

        x = torch.relu(self.conv3(x))
        print(f"  C5 Conv + ReLU  → {list(x.shape)}")

        x = x.view(x.size(0),-1)

        x = torch.relu(self.fc1(x))
        print(f"  F6 FC + ReLU    → {list(x.shape)}")

        x = self.fc2(x)
        print(f"  Output FC       → {list(x.shape)}")
        return x

if __name__ == "__main__":
    model = LeNet5(num_classes=10)
    dummy_input = torch.randn(1,1,32,32)

    print("=" * 50)
    print(f"输入形状: {list(dummy_input.shape)}")
    print("=" * 50)

    output = model(dummy_input)

    print("=" * 50)
    print(f"{'层名':<20} {'参数数量':>10}")
    print("-" * 32)
    total_params = 0
    for name, param in model.named_parameters():
        num = param.numel()
        total_params += num
        print(f"  {name:<18} {num:>10,}")
    print("-" * 32)
    print(f"  {'总参数量':<16} {total_params:>10,}")
    print("=" * 50)

输入形状: [1, 1, 32, 32]
  C1 Conv + ReLU  → [1, 6, 28, 28]
  S2 AvgPool      → [1, 6, 14, 14]
  C3 Conv + ReLU  → [1, 16, 10, 10]
  S4 AvgPool      → [1, 16, 5, 5]
  C5 Conv + ReLU  → [1, 120, 1, 1]
  F6 FC + ReLU    → [1, 84]
  Output FC       → [1, 10]
层名                         参数数量
--------------------------------
  conv1.weight              150
  conv1.bias                  6
  conv2.weight            2,400
  conv2.bias                 16
  conv3.weight           48,000
  conv3.bias                120
  fc1.weight             10,080
  fc1.bias                   84
  fc2.weight                840
  fc2.bias                   10
--------------------------------
  总参数量                 61,706


In [1]:
import torch
import torch.nn as nn
import math

class LinearModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.linear = nn.Linear(10,1)
    def forward(self,x):
        return self.linear(x)

In [2]:
class Demo(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv = nn.Conv2d(3,16,3)
        self.weight = nn.Parameter(torch.randn(10,5))
        self.register_buffer('running_mean',torch.zeros(10))
        self.threshold = 0.5
demo = Demo()
print(demo._modules.keys())
print(demo._parameters.keys())
print(demo._buffers.keys())

dict_keys(['conv'])
dict_keys(['weight'])
dict_keys(['running_mean'])


In [4]:
class LinearModel(nn.Module):
    def __init__(self,in_features,out_features,bias=True):
        super().__init__()
        self.weight = nn.Parameter(torch.empty(out_features,in_features))
        if bias:
            self.bias = nn.Parameter(torch.empty(out_features))
        self.reset_parameters()
    def forward(self,input):
        return F.linear(input,self.weight,self.bias)
